## **Feature Engineering**

In this project, feature engineering was used to transform the raw customer data into a format suitable for machine learning models. This included converting categorical variables such as contract type and payment method into numerical representations, standardizing values for consistency, and removing irrelevant or redundant features that could negatively impact model performance. These steps ensured that the models could effectively learn patterns related to customer churn, leading to more accurate predictions and meaningful insights.

<br>

#### **1. Setup & Import Libraries**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

<br>

#### **2. Load Data**

In [2]:
df = pd.read_csv("../data/raw/Post_EDA.csv")

df.head()

,Country,State,City,Zip Code,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,United States,California,Los Angeles,90003,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,United States,California,Los Angeles,90005,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,United States,California,Los Angeles,90006,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,United States,California,Los Angeles,90010,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,United States,California,Los Angeles,90015,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


<br>

#### **3. Define Target Variable**

In [3]:
target = 'Churn Value'

<br>

#### **4. Drop Unnecessary / Leakage Columns**

In [4]:
df.drop(columns=[
    'Churn Label',    # duplicate target
    'Churn Score',     # data leakage
    'Churn Reason'    # data leakage
], inplace=True, errors='ignore')

<br>

#### **5. Drop High-Cardinality / Low-Value Columns**

In [5]:
df.drop(columns=[
    'City',
    'Zip Code'
], inplace=True, errors='ignore')

<br>

#### **6. Convert Yes/No Columns → 1/0**

In [6]:
yes_no_cols = df.select_dtypes(include='object').columns

# Preview unique values
for col in yes_no_cols:
    print(col, df[col].unique())

Country ['United States']
State ['California']
Gender ['Male' 'Female']
Senior Citizen ['No' 'Yes']
Partner ['No' 'Yes']
Dependents ['No' 'Yes']
Phone Service ['Yes' 'No']
Multiple Lines ['No' 'Yes' 'No phone service']
Internet Service ['DSL' 'Fiber optic' 'No']
Online Security ['Yes' 'No' 'No internet service']
Online Backup ['Yes' 'No' 'No internet service']
Device Protection ['No' 'Yes' 'No internet service']
Tech Support ['No' 'Yes' 'No internet service']
Streaming TV ['No' 'Yes' 'No internet service']
Streaming Movies ['No' 'Yes' 'No internet service']
Contract ['Month-to-month' 'Two year' 'One year']
Paperless Billing ['Yes' 'No']
Payment Method ['Mailed check' 'Electronic check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


In [7]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].replace({
            'No internet service': 'No',
            'No phone service': 'No'
        })

In [8]:
binary_map = {'Yes': 1, 'No': 0}

for col in df.columns:
    if df[col].dtype == 'object':
        if set(df[col].unique()).issubset({'Yes', 'No'}):
            df[col] = df[col].map(binary_map)

<br>

#### **7. One-Hot Encode Remaining Categorical Columns**

In [9]:
df = pd.get_dummies(df, drop_first=True)

df.head()

df = df.astype(int)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 25 columns):
 #   Column                                  Non-Null Count  Dtype
---  ------                                  --------------  -----
 0   Senior Citizen                          7043 non-null   int32
 1   Partner                                 7043 non-null   int32
 2   Dependents                              7043 non-null   int32
 3   Tenure Months                           7043 non-null   int32
 4   Phone Service                           7043 non-null   int32
 5   Multiple Lines                          7043 non-null   int32
 6   Online Security                         7043 non-null   int32
 7   Online Backup                           7043 non-null   int32
 8   Device Protection                       7043 non-null   int32
 9   Tech Support                            7043 non-null   int32
 10  Streaming TV                            7043 non-null   int32
 11  Streaming Movies 

<br>

#### **8. Separate Features and Target**

In [11]:
X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (7043, 24)
Target shape: (7043,)


<br>

#### **9. Train-Test Split**

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5634, 24)
Test shape: (1409, 24)


<br>

#### **10. Save Processed Data for Backup**

In [13]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

<br><br>

## **Summary**
- Converted binary categorical variables (Yes/No → 1/0)
- Applied one-hot encoding to multi-category features
- Removed high-cardinality and non-informative features (City, Zip Code)
- Eliminated data leakage variables (Churn Score)
- Prepared dataset for machine learning modeling